# 🧮 SymPy Issue #30900 · Missing Eigenvalue Fix
### Match Zero Official Technical Verification Notebook
**Reported Defect:** [SymPy Issue #30900 (M3-1 & M3-2)](https://github.com/sympy/sympy/issues/30900)  
**Shared Fork:** [arushigarg2709-git/sympy](https://github.com/arushigarg2709-git/sympy)  
**Branch:** `fix/eigenvals-missing-multiplicity-jordan`

---
This notebook allows judges to verify the defect and resolution with **1-click execution** in Google Colab without installing anything locally.

## 1. Setup & Environment Verification

In [ ]:
!git clone https://github.com/arushigarg2709-git/sympy.git -b fix/eigenvals-missing-multiplicity-jordan
%cd sympy
!pip install -q mpmath pytest hypothesis
import sys
sys.path.insert(0, '.')
import sympy as sp
print('✔ Loaded SymPy from:', sp.__file__)
print('✔ SymPy Version:', sp.__version__)

## 2. Live Verification of Defect M3-1 (Preserving Multiplicities)
**Mathematical Contract:** For $A = \text{diag}(\sin^2(x), \sin^2(x), 1 - \cos^2(x), 7)$, the algebraic multiplicities must sum to $4$.

In [ ]:
from sympy import Matrix, Symbol, sin, cos
x = Symbol('x')
a = sin(x)**2
b = 1 - cos(x)**2
M = Matrix.diag(a, a, b, 7)

# Execute public SymPy API call
eigs = M.eigenvals(simplify=True)
print('Computed Eigenvalues:', eigs)
print('Algebraic Multiplicities Sum:', sum(eigs.values()), '/ 4')

assert eigs == {sin(x)**2: 3, 7: 1}, 'Defect M3-1 violated!'
print('🎉 INVARIANT PRESERVED: 100% of roots conserved without overwrite!')

## 3. Live Verification of Defect M3-2 (Jordan Invertibility & Similarity)
**Mathematical Contract:** Transformation matrix $P$ must be non-singular ($\det(P) \neq 0$) and $M P = P J$.

In [ ]:
from sympy import simplify
M2 = Matrix([[a, 1], [0, b]])
assert M2.is_diagonalizable() is False

P, J = M2.jordan_form()
det_P = simplify(P.det())
diff = simplify(M2 * P - P * J)

print('Jordan Canonical Basis P:', P)
print('det(P):', det_P)
print('Similarity Check MP - PJ == 0:', diff == Matrix.zeros(2, 2))

assert det_P != 0, 'Defect M3-2: P is singular!'
assert diff == Matrix.zeros(2, 2), 'Similarity violated!'
print('🎉 INVARIANT PRESERVED: Jordan basis is full-rank and invertible!')

## 4. Execute Full SymPy Eigenvalue Regression Suite

In [ ]:
!pytest sympy/matrices/tests/test_eigen.py -v